# Pulse AI RCM A/B Testing & Power Analysis

This notebook analyzes the A/B testing results comparing two model variants:
- **Control Group (A)**: `PulseCoder-v1.8`
- **Treatment Group (B)**: `PulseCoder-v2.1`

We will evaluate three key metrics:
1. **AI Automation Rate** (percentage of claims auto-billed without human audit)
2. **AI Coding Accuracy** (how often predicted codes match ground truth)
3. **Claim Denial Rate** (percentage of submitted claims denied by payers)

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.power import NormalIndPower

sns.set_theme(style="whitegrid")

## 1. Load Data from SQLite

In [ ]:
conn = sqlite3.connect('../pulse_ai.db')

# Fetch joined telemetry data
query = """
SELECT 
    e.encounter_id,
    e.specialty,
    e.charge_amount,
    ai.ai_model_version,
    ai.confidence_score,
    ai.action_taken,
    (ai.predicted_icd10 = e.correct_icd10 AND ai.predicted_cpt = e.correct_cpt) as is_accurate,
    c.status as claim_status,
    c.denial_reason,
    c.paid_amount
FROM encounters e
JOIN ai_coding_logs ai ON e.encounter_id = ai.encounter_id
JOIN claims c ON e.encounter_id = c.encounter_id
"""
df = pd.read_sql_query(query, conn)
conn.close()

df.head()

## 2. Metric Summaries by Model Version

In [ ]:
summary = df.groupby('ai_model_version').agg(
    total_claims=('encounter_id', 'count'),
    auto_billed=('action_taken', lambda x: (x == 'auto_billed').sum()),
    accurate_predictions=('is_accurate', 'sum'),
    denied_claims=('claim_status', lambda x: (x == 'denied').sum()),
    avg_confidence=('confidence_score', 'mean'),
    total_charges=('charge_amount', 'sum'),
    total_paid=('paid_amount', 'sum')
).reset_index()

summary['automation_rate'] = summary['auto_billed'] / summary['total_claims']
summary['accuracy_rate'] = summary['accurate_predictions'] / summary['total_claims']
summary['denial_rate'] = summary['denied_claims'] / summary['total_claims']
summary['leakage_rate'] = 1 - (summary['total_paid'] / summary['total_charges'])

summary

## 3. Statistical Significance Testing

In [ ]:
# Filter out groups
group_a = df[df['ai_model_version'] == 'PulseCoder-v1.8']
group_b = df[df['ai_model_version'] == 'PulseCoder-v2.1']

print("=== A/B Test Proportions Z-Test ===")

# Automation Rate Comparison
count_auto = [ (group_a['action_taken'] == 'auto_billed').sum(), (group_b['action_taken'] == 'auto_billed').sum() ]
nobs = [ len(group_a), len(group_b) ]
stat_auto, pval_auto = proportions_ztest(count_auto, nobs)
print(f"Automation Rate Difference p-value: {pval_auto:.4f} (Significant: {pval_auto < 0.05})")

# Accuracy Comparison
count_acc = [ group_a['is_accurate'].sum(), group_b['is_accurate'].sum() ]
stat_acc, pval_acc = proportions_ztest(count_acc, nobs)
print(f"Accuracy Rate Difference p-value: {pval_acc:.4f} (Significant: {pval_acc < 0.05})")

# Denial Rate Comparison
count_den = [ (group_a['claim_status'] == 'denied').sum(), (group_b['claim_status'] == 'denied').sum() ]
stat_den, pval_den = proportions_ztest(count_den, nobs)
print(f"Denial Rate Difference p-value: {pval_den:.4f} (Significant: {pval_den < 0.05})")

## 4. Power Analysis for Future Tests

In [ ]:
# Find required sample size for a future test if we want to detect a 3% change in denial rate (e.g. from 20% to 17%)
p1 = 0.20
p2 = 0.17
effect_size = 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))

power_analysis = NormalIndPower()
sample_size = power_analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.80, ratio=1.0)
print(f"Required sample size per variant to detect 3% reduction in denial rate: {int(np.ceil(sample_size))}")

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence Scores Density
sns.kdeplot(data=df, x='confidence_score', hue='ai_model_version', fill=True, ax=axes[0], palette='viridis')
axes[0].set_title('AI Confidence Distribution by Model')
axes[0].set_xlabel('Confidence Score')

# Denial Rate comparison
sns.barplot(data=summary, x='ai_model_version', y='denial_rate', ax=axes[1], palette='muted')
axes[1].set_title('Claim Denial Rate comparison')
axes[1].set_ylabel('Denial Rate')
axes[1].set_xlabel('Model Version')

plt.tight_layout()
plt.show()